In [ ]:
!pip install pysam

In [ ]:
from tqdm import tqdm
import pysam

In [ ]:
b = 0

In [ ]:
inds_trios = {}

with open(f'/mnt/project/DNM/trios/blocks/trios_pairs_b{b}.txt') as F:
            
    for i, row in enumerate(F):
                
        row = row.strip()
        row = row.split(' ')
    
        child = str(row[0])
        father = str(row[1])
        mother = str(row[2])
            
        child_idx = f'{child}_b{b}_p{i}'
        father_idx = f'{father}_b{b}_p{i}'
        mother_idx = f'{mother}_b{b}_p{i}'
            
        inds_trios[child_idx] = (father_idx, mother_idx)

In [ ]:
inds_diffs = {}

for chrom in range(1, 23):

    path = f'/mnt/project/DNM/trios/diffs/chr{chrom}/'

    with open(f'/mnt/project/DNM/trios/blocks/trios_pairs_b{b}.txt') as F:

        for i, _ in enumerate(F):

            # Remove trio if child is missing
            cnt = 0

            with open(f'{path}trios_chr{chrom}_b{b}_p{i}.tsv') as f:

                for row in f:

                    row = row.strip()
                    row = row.split()
                    cnt += 1 if len(row) > 0 else 0

            if cnt < 1:
                continue

            # Record differences per child
            with open(f'{path}trios_chr{chrom}_b{b}_p{i}.tsv') as f:

                for row in f:

                    row = row.strip()
                    row = row.split()

                    ind = str(row[0].split('_')[1])
                    ind_idx = f'{ind}_b{b}_p{i}'
                    if ind_idx not in inds_diffs:
                        inds_diffs[ind_idx] = []

                    ind_diffs = row[1].split('|')[1:] if len(row) == 2 else []

                    for ind_diff in ind_diffs:
                        inds_diffs[ind_idx].append(ind_diff)

In [ ]:
inds_path = {}

# paths = '/mnt/project/Bulk/DRAGEN WGS/Whole genome variant call files (GVCFs) (DRAGEN) [500k release]/'

for ind_idx in inds_trios:
    
    ind = str(ind_idx.split('_')[0])
    # inds_path[ind_idx] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[ind_idx] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

    ind = str(inds_trios[ind_idx][0].split('_')[0])
    # inds_path[inds_trios[ind_idx][0]] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[inds_trios[ind_idx][0]] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

    ind = str(inds_trios[ind_idx][1].split('_')[0])
    # inds_path[inds_trios[ind_idx][1]] = f'{paths}{ind[:2]}/{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'
    inds_path[inds_trios[ind_idx][1]] = f'{ind}_24051_0_0.dragen.hard-filtered.gvcf.gz'

In [ ]:
inds_diffs_qc = {}
inds_diffs_errors = {}

for _, ind_idx in tqdm(enumerate(inds_diffs)):
    
    # Father --------------------------------
    
    father_idx = inds_trios[ind_idx][0]
    
    try:
        vcf = pysam.VariantFile(inds_path[father_idx])
    except:
        continue # failed to read gVCF

    father_info = {}
    
    for diff_id in inds_diffs[ind_idx]:
        
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                
                father_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)
    
    # Mother --------------------------------
    
    mother_idx = inds_trios[ind_idx][1]
    
    try:
        vcf = pysam.VariantFile(inds_path[mother_idx])
    except:
        continue # failed to read gVCF

    mother_info = {}
    
    for diff_id in inds_diffs[ind_idx]:
        
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                
                mother_info[diff_id] = (ref_ad, alt_ad, dp, gq, alleles)
    
    # Individual --------------------------------
    
    try:
        vcf = pysam.VariantFile(inds_path[ind_idx])
    except:
        continue # failed to read gVCF
    
    inds_diffs_qc[ind_idx] = []
    inds_diffs_errors[ind_idx] = []
    
    for diff_id in inds_diffs[ind_idx]:

        if (diff_id not in father_info or
            diff_id not in mother_info):
            continue
            
        chrom = int(diff_id.split(':')[0])
        pos = int(diff_id.split(':')[1])
        ref = diff_id.split(':')[2]
        alt = diff_id.split(':')[3]
            
        for F in vcf.fetch(f'chr{chrom}', pos-1, pos, reopen = True):
            
            for f in list(F.samples):
            
                dp = F.samples[f]['DP']
                gq = F.samples[f]['GQ']
                gt = F.samples[f]['GT']
                
                if None in gt:
                    continue
                
                if F.pos == pos:
                    
                    ref_ad = 0
                    alt_ad = 0
                    
                    if ref in F.alleles:
                        i = F.alleles.index(ref)
                        ref_ad = F.samples[f]['AD'][i]
                        
                    if alt in F.alleles:
                        i = F.alleles.index(alt)
                        alt_ad = F.samples[f]['AD'][i]
                        
                    alleles = (F.alleles[gt[0]], F.alleles[gt[1]])
                        
                else:
                    ref_ad = dp
                    alt_ad = 0
                    alleles = (ref, ref)
                    
                info = (diff_id, ref_ad, alt_ad, dp, gq, alleles,
                        father_info[diff_id][0], father_info[diff_id][1], father_info[diff_id][2], father_info[diff_id][3], father_info[diff_id][4],
                        mother_info[diff_id][0], mother_info[diff_id][1], mother_info[diff_id][2], mother_info[diff_id][3], mother_info[diff_id][4])
                        
                if ((ref in alleles and alt in alleles) and
                    (father_info[diff_id][4][0] == father_info[diff_id][4][1] and ref in father_info[diff_id][4]) and
                    (mother_info[diff_id][4][0] == mother_info[diff_id][4][1] and ref in mother_info[diff_id][4])):
                    inds_diffs_qc[ind_idx].append(info)
                else:
                    inds_diffs_errors[ind_idx].append(info)

In [ ]:
with open(f'trios_qc_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_qc:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, father_ref_ad, father_alt_ad, father_dp, father_gq, father_alleles, mother_ref_ad, mother_alt_ad, mother_dp, mother_gq, mother_alleles) in inds_diffs_qc[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {father_ref_ad} {father_alt_ad} {father_dp} {father_gq} {father_alleles} {mother_ref_ad} {mother_alt_ad} {mother_dp} {mother_gq} {mother_alleles}\n')

In [ ]:
with open(f'trios_errors_b{b}.txt', 'w') as f:
    for ind_idx in inds_diffs_errors:
        for (diff_id, ref_ad, alt_ad, dp, gq, alleles, father_ref_ad, father_alt_ad, father_dp, father_gq, father_alleles, mother_ref_ad, mother_alt_ad, mother_dp, mother_gq, mother_alleles) in inds_diffs_errors[ind_idx]:
            f.write(f'{ind_idx} {diff_id} {ref_ad} {alt_ad} {dp} {gq} {alleles} {father_ref_ad} {father_alt_ad} {father_dp} {father_gq} {father_alleles} {mother_ref_ad} {mother_alt_ad} {mother_dp} {mother_gq} {mother_alleles}\n')